In [1]:
# bowaka_v2_lab notebook bootstrap cell — DO NOT EDIT BY HAND.
# Adds the lab's src/ (and its bowaka_common dependency) to sys.path and pins
# the working directory to the repo root, so `import bowaka_v2_lab` and
# repo-root-relative CONFIG_PATH parameters resolve identically under jupyter,
# papermill, and the QuantsLab scheduler.
import os
import sys
import time
from pathlib import Path

start_time = time.perf_counter()

_lab_root = None
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "src" / "bowaka_v2_lab" / "__init__.py").is_file():
        _lab_root = _candidate
        break
if _lab_root is None:
    raise RuntimeError(
        f"bowaka_v2_lab bootstrap: src/bowaka_v2_lab/ not found at or above {Path.cwd()}"
    )

# Pin CWD to the repo root so repo-root-relative CONFIG_PATH values resolve
# regardless of how the notebook was launched (jupyter CWD = notebook dir,
# scheduler = repo root). The lab ALWAYS lives at
# ``<repo_root>/research_notebooks/bowaka_v2_lab``, so derive the repo root from
# that canonical layout. A marker-only heuristic ("a dir with research_notebooks/
# AND Makefile") mis-matches the lab dir itself when it carries a Makefile and a
# stray nested research_notebooks/ — which then chdir's one level too deep and
# breaks every repo-root-relative path.
if _lab_root.parent.name == "research_notebooks":
    _repo_root = _lab_root.parent.parent
else:
    # Fallback: the repo root holds research_notebooks/bowaka_common (the sibling
    # package) — a marker a stray nested research_notebooks/ inside the lab lacks.
    _repo_root = _lab_root
    for _candidate in [_lab_root, *_lab_root.parents]:
        if (_candidate / "research_notebooks" / "bowaka_common").is_dir():
            _repo_root = _candidate
            break
os.chdir(_repo_root)

# Make the lab and its bowaka_common dependency importable from the working
# tree, even when the packages are not pip-installed. v1 bowaka_lab is
# deliberately excluded — v2 must not import v1.
for _src in (_lab_root / "src",
             _repo_root / "research_notebooks" / "bowaka_common" / "src"):
    if _src.is_dir() and str(_src) not in sys.path:
        sys.path.insert(0, str(_src))

import bowaka_v2_lab  # noqa: F401
print(f"bowaka_v2_lab {bowaka_v2_lab.__version__} (cwd={_repo_root})")


bowaka_v2_lab 0.1.0 (cwd=/quants-lab)


In [2]:
# Papermill parameters.
# NOTE: the prior bowaka_v2_walkforward_optuna.yml is QUARANTINED (realism
# audit 2026-05-22 §P0-001). The default below points at the Phase-8
# walk-forward-purpose contract-parity config that matches today's lake
# (IEX-only, current_code_parity). FEED='auto' upgrades simulation.mode
# to intended_realism the moment SIP bars+quotes land in the lake.
# Walk-forward sizing is locked to the operator spec (2026-05-23):
# train=21, val=1, final_holdout=5 -> 3 folds over the ~29.8-month IEX lake.
#
# Speedup report v2 §4 P2 / Phase 2 — default points at the workstation
# overlay (192 GiB / 18-core box; reserve 62 GiB; strict_parallel=true;
# max_workers=8). The base optuna config carries the more conservative
# reserve_system_gib=32 / strict_parallel=false defaults; flip
# CONFIG_PATH below if you intentionally want those.
# CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_actual_iex_current_code_optuna.workstation.yml'
# CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/bowaka_v2_actual_iex_current_code_optuna.workstation.matrix.yml'
CONFIG_PATH = 'research_notebooks/bowaka_v2_lab/configs/_local_container_matrix.yml'
N_TRIALS = 5000          # None -> optuna.n_trials from the config; or set an integer
N_STARTUP_TRIALS = 500  # None -> optuna.n_startup_trials; random trials before TPE
N_JOBS = None            # None -> optuna.n_jobs (Phase 5 default: 8 workers on PostgreSQL) currently at 10
FEED = 'auto'            # 'auto' (SIP > IEX > synthetic) | 'sip' | 'iex' | 'synthetic'
# Phase 10 default-on: trial 0 is pinned to the actual-contract
# parameter set (the live config) so the optimizer's best can be
# compared against the live incumbent. Set False to disable.
INCUMBENT_TRIAL = True


# 10 - Walk-Forward Optuna

Runs a **real** walk-forward parameter optimization against the shared
market-data lake. Each Optuna trial samples a parameter set, applies it
to the config, and runs a real backtest over every walk-forward
validation window; the trial objective is the median fold score. The
final-holdout window is never read during tuning.

**Feed auto-selection (`FEED='auto'`):** the notebook probes the lake
and adapts the run to the best data available -

| Lake holds | feed | simulation.mode |
|---|---|---|
| SIP bars + SIP quotes | `sip` | `intended_realism` |
| SIP bars, no SIP quotes | `sip` | `current_code_parity` |
| IEX bars only | `iex` | `current_code_parity` |
| neither | - | `smoke_fixture` (synthetic) |

Set `FEED` to `sip` / `iex` / `synthetic` to override the probe.

**Parameters:** `N_TRIALS` is the total trial count (`None` -> the
config's `optuna.n_trials`). `N_STARTUP_TRIALS` is how many of those are
random-sampling trials before TPE-guided search begins.

**Compute:** a run is `N_TRIALS` x `n_folds` real backtests - the config
default is a multi-day job. Set a small `N_TRIALS` for a quick run.

In [3]:
# Probe the lake and adapt the config's feed + simulation.mode.
from bowaka_v2_lab.optuna.autoconfig import resolve_walkforward_config
resolved = resolve_walkforward_config(CONFIG_PATH, feed_override=FEED)
print(f'feed   : {resolved.feed}')
print(f'mode   : {resolved.mode}')
print(f'reason : {resolved.reason}')
print(f'config : {resolved.path}')


feed   : sip
mode   : current_code_parity
reason : SIP bars present but no SIP quotes — current_code_parity (zero-spread quote fallback)
config : research_notebooks/bowaka_v2_lab/artifacts/resolved_configs/walkforward_resolved.yml


In [ ]:
import json
from bowaka_v2_lab.optuna.walkforward_runner import run_walkforward_study
# Realism remediation 2 Phase 8 (audit §P0-011): current_code_parity
# studies require an explicit opt-in. ResolvedWalkforwardConfig
# carries the auto-opt-in flags when the lake forces parity mode
# (IEX-only / SIP-bars-no-quotes); the mechanical cap is research_only.
#
# skip_best_trial_report=True: skip the study's internal single-best neighbour
# sweep (it re-runs folds in full artifact mode -> DQ not cached -> slow, and the
# holdout it would score needs a holdout-scope matrix that isn't built). The
# Top-N robustness sweep cell below does the finalist evaluation + report + YAMLs.
result = run_walkforward_study(
  resolved.path, n_trials=N_TRIALS, n_jobs=N_JOBS,
  n_startup_trials=N_STARTUP_TRIALS, allow_smoke=resolved.allow_smoke,
  allow_current_code_parity_study=resolved.allow_current_code_parity_study,
  tier=resolved.tier,
  incumbent_trial=INCUMBENT_TRIAL,
  skip_best_trial_report=True,
)
print(json.dumps(result, indent=2, default=str))

2026-06-05 21:00:20,417 INFO preflight passed: 4 checks
2026-06-05 21:09:36,215 INFO full per-fold preflight passed (mode=current_code_parity): 4 folds, 16 checks, 1 partial-tape limitation(s): ['missing_historical_quotes']
/quants-lab/research_notebooks/bowaka_v2_lab/src/bowaka_v2_lab/optuna/dispatcher.py:78: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler=optuna.samplers.TPESampler(
[I 2026-06-05 21:09:36,640] Using an existing study with name 'bowaka_v2_sip_walkforward_conservative_69a49b0a_20260605' instead of creating a new one.
2026-06-05 21:09:36,692 INFO walk-forward study bowaka_v2_sip_walkforward_conservative_69a49b0a_20260605: 5000 trials (500 random startup) x 3 folds, feed=sip, search_space_version=3; per-fold universe = daily point-in-time set (~1903 eligible on 2025-08-27) (preflight probe sampled 100 symbols)
2026-06-05 21:09:36,733 INFO incumbent baseline: enqueued trial 0 with 30 params (all 

# Top-N finalist robustness + holdout sweep

The study above optimises the **objective** (v2 = log-return **minus** edge / turnover / fill penalties, so it can be negative even when PnL is positive). The single best trial by objective is not necessarily the best config to ship — a high score can sit on a *fragile spike* that collapses under a small parameter change, and out-of-sample behaviour matters more than the dev score.

This cell evaluates the **top-N finalists** side by side so you can pick a deployable config yourself:

- **Neighbour / robustness sweep** — for each finalist, re-score `NEIGHBOURS` parameter sets perturbed ±10% on the validation folds (flat plateau vs fragile spike).
- **Final-holdout** — score each finalist on the reserved holdout window (skipped gracefully if its scan-matrix isn't built).
- **PnL + full quant metrics** — net return, max drawdown, worst-day loss, win rate, avg/median trade return, # trades, fill rate, per-fold detail.
- Writes a **markdown comparison report** (rendered inline below) + a JSON sidecar + **two single-best YAMLs**: the robustness winner and the study's #1 by objective.

It runs `scripts/topn_robustness_sweep.py` as a subprocess (fork-parallel is unsafe inside a Jupyter kernel) against the **resolved** config the study used. Tune `TOP_N` / `NEIGHBOURS` / `JOBS` below. Budget ~30–50 min (building the fold contexts once dominates; the parallel sweep is fast).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

# --- knobs ------------------------------------------------------------------
TOP_N = 12          # finalists to evaluate (ranked by median fold score − variance penalty)
NEIGHBOURS = 7      # ±10% perturbation neighbours per finalist
JOBS = None         # parallel workers; None -> min(TOP_N, cpu-2)
# Which study: None -> auto-detect the most recent walk-forward study in storage
# (the one cell 4 just produced). Set a name to target a specific finished study.
STUDY_NAME = None

_script = "research_notebooks/bowaka_v2_lab/scripts/topn_robustness_sweep.py"
_out = "research_notebooks/bowaka_v2_lab/artifacts/optuna/topn_robustness.md"
# Use the RESOLVED config the study ran against (same plan / lake / lineage).
_cfg = resolved.path  # noqa: F821 (resolved is defined in the lake-probe cell above)

_cmd = [sys.executable, _script, "--config", str(_cfg),
        "--top-n", str(TOP_N), "--neighbours", str(NEIGHBOURS), "--out", _out]
if JOBS:
    _cmd += ["--jobs", str(JOBS)]
if STUDY_NAME:
    _cmd += ["--study-name", STUDY_NAME]

# The script imports the lab from the working tree; pass PYTHONPATH to the child.
_env = dict(os.environ)
_pp = os.pathsep.join(["research_notebooks/bowaka_v2_lab/src",
                       "research_notebooks/bowaka_common/src"])
_env["PYTHONPATH"] = _pp + (os.pathsep + _env["PYTHONPATH"] if _env.get("PYTHONPATH") else "")

print("Top-N robustness + holdout sweep — builds fold contexts once, then sweeps finalists in parallel.")
print("(~30-50 min; progress streams below.)\n", flush=True)
_proc = subprocess.Popen(_cmd, cwd=os.getcwd(), env=_env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
for _line in _proc.stdout:
    print(_line, end="")
_proc.wait()

if _proc.returncode == 0 and Path(_out).exists():
    print("\n" + "=" * 72 + "\nReport (also at " + _out + "):\n")
    display(Markdown(Path(_out).read_text(encoding="utf-8")))
    for _y in sorted(Path(_out).parent.glob(Path(_out).stem + "_*.yml")):
        print("exported config YAML:", _y)
else:
    print(f"\nsweep exited with code {_proc.returncode} — see the output above.")

In [ ]:
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Script finished in {execution_time:.4f} seconds.")

execution_minutes = execution_time / 60
print(f"Script finished in {execution_minutes:.2f} minutes.")